## About Dataset
Context
This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content
5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements
This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

#### Best Practises
1. Preprocessing And Cleaning
2. Train Test Split
3. BOW,TFIDF,Word2vec
4. Train ML algorithms

In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
# Load the datset
import pandas as pd
data=pd.read_csv('all_kindle_review.csv')

In [3]:
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
df=data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [5]:
df.shape

(12000, 2)

In [6]:
## Missing values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [7]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [8]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

## Preprocessing and Cleaning

In [9]:
# positive review is 1 and negative review is 0
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)

C:\Users\saksh\AppData\Local\Temp\ipykernel_7696\4242987784.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)


In [10]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [11]:
## 1.lower all the cases
df['reviewText']=df['reviewText'].str.lower()

C:\Users\saksh\AppData\Local\Temp\ipykernel_7696\1397059194.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].str.lower()


In [12]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [13]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from bs4 import BeautifulSoup

In [16]:
## removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-Z 0-9-]+', '', x))
## removing the stop words
df['reviewText']=df['reviewText'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords.words('english')]))
## removing urls
df['reviewText']=df['reviewText'].apply(lambda x:re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## removing html tags
df['reviewText']=df['reviewText'].apply(lambda x:BeautifulSoup(x, 'lxml').get_text())
## removing extra spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

C:\Users\saksh\AppData\Local\Temp\ipykernel_7696\312417842.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-Z 0-9-]+', '', x))
C:\Users\saksh\AppData\Local\Temp\ipykernel_7696\312417842.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords.words('english')]))
C:\Users\saksh\AppData\Local\Temp\ipykernel_7696\312417842.py:6: SettingWi

In [18]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [19]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer

In [20]:
lemmatizer = WordNetLemmatizer()

In [21]:
def lemmatize_words(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

In [22]:
df['reviewText'] = df['reviewText'].apply(lambda x:lemmatize_words(x))

C:\Users\saksh\AppData\Local\Temp\ipykernel_7696\3509282327.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText'] = df['reviewText'].apply(lambda x:lemmatize_words(x))


In [23]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [24]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.2, random_state=42)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
bow= CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [27]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [28]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow= GaussianNB().fit(X_train_bow, y_train)
nb_model_tfidf= GaussianNB().fit(X_train_tfidf, y_train)

In [29]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [30]:
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [31]:
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [32]:
print("Accuracy for BOW model: ", accuracy_score(y_test, y_pred_bow))
print("Classification Report for BOW model:\n", classification_report(y_test, y_pred_bow))
print("Confusion Matrix for BOW model:\n", confusion_matrix(y_test, y_pred_bow))

Accuracy for BOW model:  0.5745833333333333
Classification Report for BOW model:
               precision    recall  f1-score   support

           0       0.41      0.62      0.49       803
           1       0.74      0.55      0.63      1597

    accuracy                           0.57      2400
   macro avg       0.58      0.59      0.56      2400
weighted avg       0.63      0.57      0.59      2400

Confusion Matrix for BOW model:
 [[499 304]
 [717 880]]


In [33]:
print("Accuracy for TF-IDF model: ", accuracy_score(y_test, y_pred_tfidf))
print("Classification Report for TF-IDF model:\n", classification_report(y_test, y_pred_tfidf))
print("Confusion Matrix for TF-IDF model:\n", confusion_matrix(y_test, y_pred_tfidf))

Accuracy for TF-IDF model:  0.57875
Classification Report for TF-IDF model:
               precision    recall  f1-score   support

           0       0.41      0.61      0.49       803
           1       0.74      0.56      0.64      1597

    accuracy                           0.58      2400
   macro avg       0.58      0.59      0.57      2400
weighted avg       0.63      0.58      0.59      2400

Confusion Matrix for TF-IDF model:
 [[488 315]
 [696 901]]


In [34]:
from gensim.models import Word2Vec
import numpy as np

# Tokenize the reviews for Word2Vec
X_train_tokens = X_train.apply(lambda x: x.split())
X_test_tokens = X_test.apply(lambda x: x.split())

# Train Word2Vec model
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=1, workers=4, seed=42)

# Function to get average word2vec embedding for a document
def document_vector(doc, model):
    # Remove out-of-vocabulary words
    words = [word for word in doc if word in model.wv]
    if len(words) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[words], axis=0)

# Create averaged word2vec vectors for train and test sets
X_train_w2v = np.vstack(X_train_tokens.apply(lambda x: document_vector(x, w2v_model)))
X_test_w2v = np.vstack(X_test_tokens.apply(lambda x: document_vector(x, w2v_model)))

In [35]:
# Train a GaussianNB model using Word2Vec features
nb_model_w2v = GaussianNB().fit(X_train_w2v, y_train)

# Predict on test set
y_pred_w2v = nb_model_w2v.predict(X_test_w2v)

# Evaluate metrics
print("Accuracy for Word2Vec model: ", accuracy_score(y_test, y_pred_w2v))
print("Classification Report for Word2Vec model:\n", classification_report(y_test, y_pred_w2v))
print("Confusion Matrix for Word2Vec model:\n", confusion_matrix(y_test, y_pred_w2v))

Accuracy for Word2Vec model:  0.6720833333333334
Classification Report for Word2Vec model:
               precision    recall  f1-score   support

           0       0.51      0.77      0.61       803
           1       0.84      0.62      0.72      1597

    accuracy                           0.67      2400
   macro avg       0.68      0.70      0.66      2400
weighted avg       0.73      0.67      0.68      2400

Confusion Matrix for Word2Vec model:
 [[619 184]
 [603 994]]
